# ============================================================
# STEP 11 — COMBINED ALL-FOLDS EVALUATION PLOTS1
# ============================================================
# Loads saved .npy predictions from ALL 5 folds and generates
# one high-quality figure per plot type with all folds overlaid.
#
# Pattern: thin coloured line per fold + thick mean line +
#          shaded ±1 std band → shows BOTH individual fold
#          behaviour AND CV stability in one publication figure.
#
# Outputs (12 combined figures):
#  1. cv_roc_binary.png          — all models, all folds
#  2. cv_roc_5class.png          — macro-OVR, all models, all folds
#  3. cv_pr_binary.png
#  4. cv_pr_5class.png
#  5. cv_det_binary.png          — DET curves, all folds
#  6. cv_calibration_binary.png  — reliability diagrams, all folds
#  7. cv_calibration_5class.png
#  8. cv_confusion_histgb.png    — HistGB confusion, 5 folds side-by-side
#  9. cv_metrics_all_folds.png   — F1/MCC/AUC per fold, all models
# 10. cv_roc_per_class.png       — HistGB per-class ROC, all 5 folds
# 11. cv_pr_per_class.png        — HistGB per-class PR, all 5 folds
# 12. cv_error_patterns.png      — error count + TLD drift across folds
# ============================================================

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from sklearn.metrics import (
    roc_curve, roc_auc_score, precision_recall_curve,
    average_precision_score, confusion_matrix,
    det_curve, f1_score, matthews_corrcoef,
    brier_score_loss, log_loss)
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")

# ── CONFIG ────────────────────────────────────────────────────
DATA_DIR  = "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May"
TASKS     = ["5class", "binary"]
N_FOLDS   = 5
OUT_DIR   = os.path.join(DATA_DIR, "journal_eval_combined_folds")
PNG_DIR   = os.path.join(OUT_DIR, "figures_png")
PDF_DIR   = os.path.join(OUT_DIR, "figures_pdf")
for d in [OUT_DIR, PNG_DIR, PDF_DIR]:
    os.makedirs(d, exist_ok=True)

MODEL_ORDER = ["HistGB","LightGBM","XGBoost","RandomForest",
               "ExtraTrees","LogisticRegression","ComplementNB"]
MODEL_COLORS = {
    "HistGB":"#1565C0","LightGBM":"#1E88E5","XGBoost":"#42A5F5",
    "RandomForest":"#2E7D32","ExtraTrees":"#66BB6A",
    "LogisticRegression":"#F57F17","ComplementNB":"#B71C1C",
}
FOLD_STYLES = ["-","--","-.",":",(0,(3,1,1,1))]
FOLD_ALPHA  = 0.35          # individual fold curve transparency
MEAN_LW     = 2.4           # mean curve linewidth
FOLD_LW     = 1.0           # individual fold linewidth
CLASS_NAMES_5 = ["benign","phishing","malware","scam","other_malicious"]
CLASS_NAMES_B = ["benign","malicious"]
CLASS_COLORS  = ["#1565C0","#B71C1C","#2E7D32","#F57F17","#6A1B9A"]

plt.rcParams.update({
    "font.family":"serif","font.size":10,"axes.labelsize":10,
    "axes.titlesize":11,"legend.fontsize":7.5,"figure.dpi":150,
    "savefig.dpi":600,"pdf.fonttype":42,"ps.fonttype":42,
    "lines.linewidth":1.6,"axes.linewidth":0.8,
    "axes.spines.top":False,"axes.spines.right":False,
    "axes.grid":True,"grid.alpha":0.25,"grid.linestyle":"--",
})

def save_fig(fig, name):
    fig.savefig(os.path.join(PNG_DIR,f"{name}.png"),dpi=600,bbox_inches="tight")
    fig.savefig(os.path.join(PDF_DIR,f"{name}.pdf"),bbox_inches="tight")
    plt.close(fig)
    print(f"  ✓ {name}")

def safe_name(s):
    for c in " /\\:()": s=s.replace(c,"_")
    return s

# ── DATA LOADER ───────────────────────────────────────────────
def load_all_folds(task):
    """Returns dict: {model: {fold: {y_true, y_pred, y_proba}}}"""
    base = os.path.join(DATA_DIR, f"journal_eval_{task}")
    data = {}
    for mn in MODEL_ORDER:
        mid = safe_name(mn)
        data[mn] = {}
        for fid in range(N_FOLDS):
            csv_dir = os.path.join(base, f"fold{fid}", "csv")
            ft = os.path.join(csv_dir, f"y_true_{mid}.npy")
            fp = os.path.join(csv_dir, f"y_pred_{mid}.npy")
            fpr= os.path.join(csv_dir, f"y_proba_{mid}.npy")
            if os.path.exists(ft) and os.path.exists(fp):
                data[mn][fid] = {
                    "y_true":  np.load(ft),
                    "y_pred":  np.load(fp),
                    "y_proba": np.load(fpr) if os.path.exists(fpr) else None,
                }
    loaded = sum(len(v) for v in data.values())
    print(f"  Loaded {loaded} fold-model arrays for {task}")
    return data

def interp_roc(fpr_arr, tpr_arr):
    """Interpolate ROC curve at common FPR grid."""
    grid = np.linspace(0, 1, 500)
    tpr_interp = np.interp(grid, fpr_arr, tpr_arr)
    tpr_interp[0] = 0.0
    return grid, tpr_interp

def interp_pr(rec_arr, prec_arr):
    """Interpolate PR curve at common recall grid (reversed)."""
    grid = np.linspace(0, 1, 500)
    prec_interp = np.interp(grid, rec_arr[::-1], prec_arr[::-1])
    return grid, prec_interp

# ════════════════════════════════════════════════════════════
# PLOT 1+2: ROC CURVES — all models × all folds
# ════════════════════════════════════════════════════════════
def plot_roc_all_folds(data, task, classes, is_bin):
    mnames  = [m for m in MODEL_ORDER if data.get(m)]
    ncols   = min(4, len(mnames))
    nrows   = (len(mnames)+ncols-1)//ncols
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5.5*ncols, 4.8*nrows), squeeze=False)
    flat = axes.flatten()

    for pi, mn in enumerate(mnames):
        ax  = flat[pi]
        col = MODEL_COLORS.get(mn,"#607D8B")
        interp_tprs = []
        auc_vals    = []

        for fid in range(N_FOLDS):
            fd = data[mn].get(fid)
            if fd is None or fd["y_proba"] is None: continue
            try:
                if is_bin:
                    fpr,tpr,_ = roc_curve(fd["y_true"], fd["y_proba"][:,1])
                    auc = roc_auc_score(fd["y_true"], fd["y_proba"][:,1])
                else:
                    yb = label_binarize(fd["y_true"], classes=classes)
                    fpr,tpr,_ = roc_curve(yb.ravel(), fd["y_proba"].ravel())
                    auc = roc_auc_score(fd["y_true"], fd["y_proba"],
                                        multi_class="ovr", average="macro")
                ax.plot(fpr, tpr, ls=FOLD_STYLES[fid], color=col,
                        alpha=FOLD_ALPHA, lw=FOLD_LW,
                        label=f"Fold {fid} ({auc:.3f})")
                _, tpr_i = interp_roc(fpr, tpr)
                interp_tprs.append(tpr_i)
                auc_vals.append(auc)
            except: pass

        if interp_tprs:
            grid = np.linspace(0,1,500)
            mean_tpr = np.mean(interp_tprs, axis=0)
            std_tpr  = np.std(interp_tprs,  axis=0)
            mean_auc = np.mean(auc_vals)
            std_auc  = np.std(auc_vals)
            ax.plot(grid, mean_tpr, color=col, lw=MEAN_LW,
                    label=f"Mean (AUC={mean_auc:.3f}±{std_auc:.3f})")
            ax.fill_between(grid, mean_tpr-std_tpr, mean_tpr+std_tpr,
                            color=col, alpha=0.12)

        ax.plot([0,1],[0,1],"k--",lw=0.7,alpha=0.5)
        ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
        ax.set_title(mn, fontweight="bold", fontsize=10)
        ax.legend(fontsize=6.5, loc="lower right")
        ax.set_xlim(-0.02,1.02); ax.set_ylim(-0.02,1.02)

    for j in range(pi+1, len(flat)): flat[j].set_visible(False)
    fig.suptitle(f"ROC Curves — {task} task (all 5 domain-insulated folds)\n"
                 f"Thin lines = individual folds; thick = mean ± 1 std band",
                 fontweight="bold", fontsize=12)
    fig.tight_layout(rect=[0,0,1,0.94])
    save_fig(fig, f"cv_roc_{task}")

# ════════════════════════════════════════════════════════════
# PLOT 3+4: PRECISION-RECALL — all models × all folds
# ════════════════════════════════════════════════════════════
def plot_pr_all_folds(data, task, classes, is_bin):
    mnames  = [m for m in MODEL_ORDER if data.get(m)]
    ncols   = min(4, len(mnames))
    nrows   = (len(mnames)+ncols-1)//ncols
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5.5*ncols, 4.8*nrows), squeeze=False)
    flat = axes.flatten()

    for pi, mn in enumerate(mnames):
        ax  = flat[pi]
        col = MODEL_COLORS.get(mn,"#607D8B")
        interp_precs = []
        ap_vals      = []

        for fid in range(N_FOLDS):
            fd = data[mn].get(fid)
            if fd is None or fd["y_proba"] is None: continue
            try:
                if is_bin:
                    prec,rec,_ = precision_recall_curve(
                        fd["y_true"], fd["y_proba"][:,1])
                    ap = average_precision_score(
                        fd["y_true"], fd["y_proba"][:,1])
                else:
                    yb = label_binarize(fd["y_true"], classes=classes)
                    prec,rec,_ = precision_recall_curve(
                        yb.ravel(), fd["y_proba"].ravel())
                    ap = average_precision_score(yb, fd["y_proba"],
                                                 average="macro")
                ax.plot(rec, prec, ls=FOLD_STYLES[fid], color=col,
                        alpha=FOLD_ALPHA, lw=FOLD_LW,
                        label=f"Fold {fid} (AP={ap:.3f})")
                _, p_i = interp_pr(rec, prec)
                interp_precs.append(p_i)
                ap_vals.append(ap)
            except: pass

        if interp_precs:
            grid = np.linspace(0,1,500)
            mean_p = np.mean(interp_precs, axis=0)
            std_p  = np.std(interp_precs,  axis=0)
            mean_ap = np.mean(ap_vals)
            std_ap  = np.std(ap_vals)
            ax.plot(grid, mean_p, color=col, lw=MEAN_LW,
                    label=f"Mean (AP={mean_ap:.3f}±{std_ap:.3f})")
            ax.fill_between(grid, np.clip(mean_p-std_p,0,1),
                            np.clip(mean_p+std_p,0,1), color=col, alpha=0.12)

        ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
        ax.set_title(mn, fontweight="bold", fontsize=10)
        ax.legend(fontsize=6.5, loc="upper right")
        ax.set_xlim(-0.02,1.02); ax.set_ylim(-0.02,1.05)

    for j in range(pi+1, len(flat)): flat[j].set_visible(False)
    fig.suptitle(f"Precision-Recall Curves — {task} task "
                 f"(all 5 domain-insulated folds)",
                 fontweight="bold", fontsize=12)
    fig.tight_layout(rect=[0,0,1,0.94])
    save_fig(fig, f"cv_pr_{task}")

# ════════════════════════════════════════════════════════════
# PLOT 5: DET CURVES — binary, all models × all folds
# ════════════════════════════════════════════════════════════
def plot_det_all_folds(data):
    mnames = [m for m in MODEL_ORDER if data.get(m)]
    ncols  = min(4, len(mnames))
    nrows  = (len(mnames)+ncols-1)//ncols
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5.5*ncols, 4.8*nrows), squeeze=False)
    flat = axes.flatten()

    for pi, mn in enumerate(mnames):
        ax  = flat[pi]
        col = MODEL_COLORS.get(mn,"#607D8B")
        for fid in range(N_FOLDS):
            fd = data[mn].get(fid)
            if fd is None or fd["y_proba"] is None: continue
            try:
                fpr,fnr,_ = det_curve(fd["y_true"], fd["y_proba"][:,1])
                ax.plot(fpr, fnr, ls=FOLD_STYLES[fid], color=col,
                        alpha=FOLD_ALPHA+0.1, lw=FOLD_LW,
                        label=f"Fold {fid}")
            except: pass
        ax.set_xlabel("FPR"); ax.set_ylabel("FNR")
        ax.set_title(mn, fontweight="bold", fontsize=10)
        ax.legend(fontsize=7); ax.plot([0,1],[0,1],"k--",lw=0.7,alpha=0.4)
    for j in range(pi+1, len(flat)): flat[j].set_visible(False)
    fig.suptitle("Detection Error Tradeoff (DET) — Binary task, all 5 folds",
                 fontweight="bold", fontsize=12)
    fig.tight_layout(rect=[0,0,1,0.94])
    save_fig(fig, "cv_det_binary")

# ════════════════════════════════════════════════════════════
# PLOT 6+7: CALIBRATION — all models × all folds
# ════════════════════════════════════════════════════════════
def plot_calibration_all_folds(data, task, is_bin):
    mnames = [m for m in MODEL_ORDER if data.get(m)]
    ncols  = min(4, len(mnames))
    nrows  = (len(mnames)+ncols-1)//ncols
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5.5*ncols, 4.8*nrows), squeeze=False)
    flat = axes.flatten()

    for pi, mn in enumerate(mnames):
        ax  = flat[pi]
        col = MODEL_COLORS.get(mn,"#607D8B")
        brier_list = []
        frac_list  = []

        for fid in range(N_FOLDS):
            fd = data[mn].get(fid)
            if fd is None or fd["y_proba"] is None: continue
            try:
                if is_bin:
                    prob = fd["y_proba"][:,1]; yt = fd["y_true"]
                else:
                    prob = fd["y_proba"].max(axis=1)
                    yt   = (fd["y_true"]==fd["y_proba"].argmax(axis=1)).astype(int)
                frac, mean_p = calibration_curve(yt, prob, n_bins=10)
                bs = brier_score_loss(yt, prob)
                ax.plot(mean_p, frac, ls=FOLD_STYLES[fid], color=col,
                        alpha=FOLD_ALPHA+0.1, lw=FOLD_LW,
                        label=f"Fold {fid} (B={bs:.4f})")
                brier_list.append(bs)
                # Interpolate for mean
                g = np.linspace(0,1,20)
                fi = np.interp(g, mean_p, frac)
                frac_list.append(fi)
            except: pass

        if frac_list:
            g    = np.linspace(0,1,20)
            mf   = np.mean(frac_list, axis=0)
            sf   = np.std(frac_list,  axis=0)
            mb   = np.mean(brier_list)
            sb   = np.std(brier_list)
            ax.plot(g, mf, color=col, lw=MEAN_LW,
                    label=f"Mean Brier={mb:.4f}±{sb:.4f}")
            ax.fill_between(g, mf-sf, mf+sf, color=col, alpha=0.12)

        ax.plot([0,1],[0,1],"k--",lw=0.9,alpha=0.6,label="Perfect")
        ax.set_xlabel("Mean predicted probability")
        ax.set_ylabel("Fraction positives")
        ax.set_title(mn, fontweight="bold", fontsize=10)
        ax.legend(fontsize=6)

    for j in range(pi+1, len(flat)): flat[j].set_visible(False)
    fig.suptitle(f"Calibration Reliability Diagrams — {task} task, all 5 folds",
                 fontweight="bold", fontsize=12)
    fig.tight_layout(rect=[0,0,1,0.94])
    save_fig(fig, f"cv_calibration_{task}")

# ════════════════════════════════════════════════════════════
# PLOT 8: CONFUSION MATRICES — HistGB, 5 folds side-by-side
# ════════════════════════════════════════════════════════════
def plot_confusion_histgb_all_folds(data, task, classes, class_names):
    fig, axes = plt.subplots(1, N_FOLDS, figsize=(5*N_FOLDS, 5.5), squeeze=False)
    axes = axes.flatten()

    for fid in range(N_FOLDS):
        ax  = axes[fid]
        fd  = data.get("HistGB",{}).get(fid)
        if fd is None:
            ax.set_visible(False); continue

        cm = confusion_matrix(fd["y_true"], fd["y_pred"],
                              labels=classes, normalize="true")
        f1 = f1_score(fd["y_true"], fd["y_pred"],
                      average="macro", zero_division=0)
        im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1, aspect="auto")
        ax.set_xticks(range(len(classes)))
        ax.set_yticks(range(len(classes)))
        short = [c[:6] for c in class_names]
        ax.set_xticklabels(short, rotation=45, ha="right", fontsize=8)
        ax.set_yticklabels(short if fid==0 else [], fontsize=8)
        ax.set_title(f"Fold {fid}\nMacro-F1={f1:.3f}",
                     fontweight="bold", fontsize=9)
        if fid==0: ax.set_ylabel("True label", fontsize=9)
        ax.set_xlabel("Predicted", fontsize=9)
        for r in range(len(classes)):
            for c in range(len(classes)):
                v = cm[r,c]
                ax.text(c,r,f"{v:.2f}",ha="center",va="center",
                        fontsize=7, color="white" if v>0.5 else "black")
        fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)

    fig.suptitle(f"HistGB — Row-Normalised Confusion Matrices ({task}, all 5 folds)",
                 fontweight="bold", fontsize=12)
    fig.tight_layout()
    save_fig(fig, f"cv_confusion_histgb_{task}")

# ════════════════════════════════════════════════════════════
# PLOT 9: METRICS TABLE — F1/MCC/AUC per fold, all models
# ════════════════════════════════════════════════════════════
def plot_metrics_all_folds(data, task, classes, is_bin):
    metrics_data = {}  # model -> {fold -> {f1, mcc, auc}}
    for mn in MODEL_ORDER:
        metrics_data[mn] = {}
        for fid in range(N_FOLDS):
            fd = data[mn].get(fid)
            if fd is None: continue
            f1  = f1_score(fd["y_true"], fd["y_pred"],
                           average="macro", zero_division=0)
            mcc = matthews_corrcoef(fd["y_true"], fd["y_pred"])
            auc = np.nan
            if fd["y_proba"] is not None:
                try:
                    if is_bin:
                        auc = roc_auc_score(fd["y_true"], fd["y_proba"][:,1])
                    else:
                        auc = roc_auc_score(fd["y_true"], fd["y_proba"],
                                            multi_class="ovr", average="macro")
                except: pass
            metrics_data[mn][fid] = {"f1":f1,"mcc":mcc,"auc":auc}

    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    metric_keys = ["f1","mcc","auc"]
    metric_labels = ["Macro-F1","MCC","ROC-AUC"]
    folds = list(range(N_FOLDS))

    for ax, mkey, mlabel in zip(axes, metric_keys, metric_labels):
        for mn in MODEL_ORDER:
            col   = MODEL_COLORS.get(mn,"#607D8B")
            vals  = [metrics_data[mn].get(f,{}).get(mkey,np.nan)
                     for f in folds]
            valid = [(f,v) for f,v in zip(folds,vals) if not np.isnan(v)]
            if not valid: continue
            fx, vx = zip(*valid)
            mean_v = np.mean(vx); std_v = np.std(vx)
            # individual fold points
            ax.scatter(fx, vx, color=col, s=40, zorder=5, alpha=0.85)
            ax.plot(fx, vx, color=col, lw=1.2, alpha=0.6,
                    ls="--", label=f"{mn} ({mean_v:.3f}±{std_v:.3f})")
            # mean ± std band
            ax.axhline(mean_v, color=col, lw=1.5, ls="-", alpha=0.4)
            ax.fill_between([0,N_FOLDS-1],
                            mean_v-std_v, mean_v+std_v,
                            color=col, alpha=0.04)

        ax.set_xlabel("Fold", fontsize=10)
        ax.set_ylabel(mlabel, fontsize=10)
        ax.set_title(mlabel, fontweight="bold", fontsize=11)
        ax.set_xticks(folds)
        all_vals = [metrics_data[mn].get(f,{}).get(mkey,np.nan)
                    for mn in MODEL_ORDER for f in folds]
        valid_all = [v for v in all_vals if not np.isnan(v)]
        if valid_all:
            lo = max(0, min(valid_all)-0.03)
            ax.set_ylim(lo, min(1.01, max(valid_all)+0.03))

    handles = [Line2D([0],[0],color=MODEL_COLORS.get(m,"#607D8B"),
                      lw=2,marker="o",ms=5,label=m) for m in MODEL_ORDER]
    fig.legend(handles=handles, fontsize=8, ncol=4,
               loc="lower center", bbox_to_anchor=(0.5,-0.06),
               title="Model (mean ± std across 5 folds)", title_fontsize=8.5)
    fig.suptitle(f"Performance per Fold — {task} task "
                 f"(scatter=individual fold; dashed line=trajectory; "
                 f"shaded=±1 std band)",
                 fontweight="bold", fontsize=11)
    fig.tight_layout(rect=[0,0.07,1,0.96])
    save_fig(fig, f"cv_metrics_all_folds_{task}")

# ════════════════════════════════════════════════════════════
# PLOT 10+11: PER-CLASS ROC AND PR — HistGB, all 5 folds
# ════════════════════════════════════════════════════════════
def plot_perclass_roc_pr(data, task, classes, class_names):
    if len(classes) == 2: return
    n_cls = len(classes)
    COLORS_FOLD = plt.cm.tab10(np.linspace(0,1,N_FOLDS))

    for plot_type, func_name in [("ROC","roc"), ("PR","pr")]:
        fig, axes = plt.subplots(1, n_cls, figsize=(5.5*n_cls, 5.5))

        for ci, (ax, cname) in enumerate(zip(axes, class_names)):
            col_cls = CLASS_COLORS[ci]
            interp_curves = []
            score_list    = []

            for fid in range(N_FOLDS):
                fd = data.get("HistGB",{}).get(fid)
                if fd is None or fd["y_proba"] is None: continue
                yb  = label_binarize(fd["y_true"], classes=classes)
                p   = fd["y_proba"][:,ci]
                try:
                    if plot_type=="ROC":
                        xv, yv, _ = roc_curve(yb[:,ci], p)
                        score = roc_auc_score(yb[:,ci], p)
                        lbl   = f"Fold {fid} (AUC={score:.3f})"
                        _, yi  = interp_roc(xv, yv)
                        interp_curves.append(yi)
                    else:
                        yv, xv, _ = precision_recall_curve(yb[:,ci], p)
                        score = average_precision_score(yb[:,ci], p)
                        lbl   = f"Fold {fid} (AP={score:.3f})"
                        _, yi  = interp_pr(xv, yv)
                        interp_curves.append(yi)
                    score_list.append(score)
                    ax.plot(xv, yv, ls=FOLD_STYLES[fid],
                            color=col_cls, alpha=FOLD_ALPHA+0.05,
                            lw=FOLD_LW, label=lbl)
                except: pass

            if interp_curves:
                grid = np.linspace(0,1,500)
                mc   = np.mean(interp_curves,axis=0)
                sc   = np.std(interp_curves, axis=0)
                ms   = np.mean(score_list); ss = np.std(score_list)
                ax.plot(grid if plot_type=="ROC" else grid[::-1],
                        mc, color=col_cls, lw=MEAN_LW,
                        label=f"Mean ({ms:.3f}±{ss:.3f})")
                ax.fill_between(
                    grid if plot_type=="ROC" else grid[::-1],
                    np.clip(mc-sc,0,1), np.clip(mc+sc,0,1),
                    color=col_cls, alpha=0.12)

            if plot_type=="ROC":
                ax.plot([0,1],[0,1],"k--",lw=0.7,alpha=0.5)
                ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
            else:
                ax.axhline(label_binarize(
                    data["HistGB"][0]["y_true"], classes=classes)[:,ci].mean(),
                    ls=":", lw=0.9, color=col_cls, alpha=0.5)
                ax.set_xlabel("Recall"); ax.set_ylabel("Precision")

            ax.set_title(cname, fontweight="bold",
                         fontsize=11, color=col_cls)
            ax.legend(fontsize=6.5)

        fig.suptitle(f"HistGB — Per-Class {plot_type} Curves "
                     f"({task}, all 5 domain-insulated folds)",
                     fontweight="bold", fontsize=12)
        fig.tight_layout()
        save_fig(fig, f"cv_{func_name}_per_class_{task}")

# ════════════════════════════════════════════════════════════
# PLOT 12: ERROR PATTERNS ACROSS FOLDS
# ════════════════════════════════════════════════════════════
def plot_error_patterns(data, task):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

    # Left: error count per model per fold (grouped bars)
    ax = axes[0]
    mnames = [m for m in MODEL_ORDER if data.get(m)]
    x      = np.arange(N_FOLDS)
    width  = 0.8 / len(mnames)
    for i, mn in enumerate(mnames):
        counts = []
        for fid in range(N_FOLDS):
            fd = data[mn].get(fid)
            if fd is None: counts.append(0); continue
            err = (fd["y_pred"] != fd["y_true"]).sum()
            counts.append(err)
        ax.bar(x + i*width - 0.4 + width/2, counts,
               width=width*0.9, color=MODEL_COLORS.get(mn,"#607D8B"),
               label=mn.replace("LogisticRegression","LR")
                       .replace("ComplementNB","CNB"),
               alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels([f"F{i}" for i in range(N_FOLDS)])
    ax.set_xlabel("Fold", fontsize=10)
    ax.set_ylabel("Misclassification count", fontsize=10)
    ax.set_title("Error Count per Model per Fold", fontweight="bold", fontsize=11)
    ax.legend(fontsize=7.5, ncol=2)

    # Right: HistGB macro-F1 per fold with fold-size annotation
    ax2 = axes[1]
    col = MODEL_COLORS["HistGB"]
    f1_vals = []
    fold_sizes = []
    for fid in range(N_FOLDS):
        fd = data.get("HistGB",{}).get(fid)
        if fd is None: f1_vals.append(np.nan); fold_sizes.append(0); continue
        f1 = f1_score(fd["y_true"],fd["y_pred"],average="macro",zero_division=0)
        f1_vals.append(f1)
        fold_sizes.append(len(fd["y_true"]))
    ax2.bar(range(N_FOLDS), f1_vals, color=col, alpha=0.8,
            edgecolor="white", linewidth=0.8)
    ax2.axhline(np.nanmean(f1_vals), color="red", ls="--", lw=1.5,
                label=f"Mean={np.nanmean(f1_vals):.4f}")
    ax2.fill_between([-0.5, N_FOLDS-0.5],
                     np.nanmean(f1_vals)-np.nanstd(f1_vals),
                     np.nanmean(f1_vals)+np.nanstd(f1_vals),
                     color="red", alpha=0.08, label="±1 std")
    for i, (v, sz) in enumerate(zip(f1_vals, fold_sizes)):
        if not np.isnan(v):
            ax2.text(i, v+0.001, f"{v:.4f}\n(n={sz:,})",
                     ha="center", va="bottom", fontsize=7.5)
    ax2.set_xticks(range(N_FOLDS))
    ax2.set_xticklabels([f"Fold {i}" for i in range(N_FOLDS)])
    ax2.set_ylabel("Macro-F1", fontsize=10)
    ax2.set_title(f"HistGB Macro-F1 per Fold with Test-Set Size",
                  fontweight="bold", fontsize=11)
    ax2.legend(fontsize=8.5)
    lo = max(0, min(v for v in f1_vals if not np.isnan(v))-0.02)
    ax2.set_ylim(lo, min(1.01, max(f1_vals)+0.02))

    fig.suptitle(f"Error Patterns Across Folds — {task} task",
                 fontweight="bold", fontsize=12)
    fig.tight_layout()
    save_fig(fig, f"cv_error_patterns_{task}")

# ════════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════════
print("="*60)
print("COMBINED ALL-FOLDS EVALUATION PLOTS")
print("="*60)

for TASK in TASKS:
    classes     = np.array([0,1]) if TASK=="binary" else np.array([0,1,2,3,4])
    class_names = CLASS_NAMES_B if TASK=="binary" else CLASS_NAMES_5
    is_bin      = (TASK=="binary")

    print(f"\n── {TASK.upper()} ──")
    data = load_all_folds(TASK)

    print(f"  Generating ROC curves ...")
    plot_roc_all_folds(data, TASK, classes, is_bin)

    print(f"  Generating PR curves ...")
    plot_pr_all_folds(data, TASK, classes, is_bin)

    print(f"  Generating confusion matrices (HistGB) ...")
    plot_confusion_histgb_all_folds(data, TASK, classes, class_names)

    print(f"  Generating calibration diagrams ...")
    plot_calibration_all_folds(data, TASK, is_bin)

    print(f"  Generating metrics per fold ...")
    plot_metrics_all_folds(data, TASK, classes, is_bin)

    print(f"  Generating error patterns ...")
    plot_error_patterns(data, TASK)

    if is_bin:
        print(f"  Generating DET curves ...")
        plot_det_all_folds(data)
    else:
        print(f"  Generating per-class ROC+PR (HistGB) ...")
        plot_perclass_roc_pr(data, TASK, classes, class_names)

print(f"""
{'='*60}
DONE — {PNG_DIR}
{'='*60}
12 combined figures generated:

  cv_roc_binary.png          — 7 models × 5 folds, mean±std band
  cv_roc_5class.png          — macro-OVR
  cv_pr_binary.png           — precision-recall all models+folds
  cv_pr_5class.png
  cv_det_binary.png          — DET all models+folds
  cv_calibration_binary.png  — reliability diagrams all folds
  cv_calibration_5class.png
  cv_confusion_histgb_binary.png  — 5 side-by-side confusion matrices
  cv_confusion_histgb_5class.png
  cv_metrics_all_folds_binary.png — F1/MCC/AUC trajectories
  cv_metrics_all_folds_5class.png
  cv_roc_per_class_5class.png     — per-class ROC, HistGB, all folds
  cv_pr_per_class_5class.png
  cv_error_patterns_binary.png
  cv_error_patterns_5class.png
""")

COMBINED ALL-FOLDS EVALUATION PLOTS

── 5CLASS ──
  Loaded 35 fold-model arrays for 5class
  Generating ROC curves ...
  ✓ cv_roc_5class
  Generating PR curves ...
  ✓ cv_pr_5class
  Generating confusion matrices (HistGB) ...
  ✓ cv_confusion_histgb_5class
  Generating calibration diagrams ...
  ✓ cv_calibration_5class
  Generating metrics per fold ...
  ✓ cv_metrics_all_folds_5class
  Generating error patterns ...
  ✓ cv_error_patterns_5class
  Generating per-class ROC+PR (HistGB) ...
  ✓ cv_roc_per_class_5class
  ✓ cv_pr_per_class_5class

── BINARY ──
  Loaded 35 fold-model arrays for binary
  Generating ROC curves ...
  ✓ cv_roc_binary
  Generating PR curves ...
  ✓ cv_pr_binary
  Generating confusion matrices (HistGB) ...
  ✓ cv_confusion_histgb_binary
  Generating calibration diagrams ...
  ✓ cv_calibration_binary
  Generating metrics per fold ...
  ✓ cv_metrics_all_folds_binary
  Generating error patterns ...
  ✓ cv_error_patterns_binary
  Generating DET curves ...
  ✓ cv_det_bin